# Baseline Custom CNN

Why a baseline at all? Without one, any number the transfer-learning model produces is just a number. With a baseline, we can say something concrete: "DenseNet121 lifted recall from X to Y on the same data." That comparison is the entire point of including this notebook in the portfolio.

The architecture is deliberately small and conventional: four convolutional blocks, global average pooling, a single hidden dense layer, sigmoid output. Trained from scratch on roughly 5,000 images, it should reach respectable accuracy but is expected to underperform the pretrained DenseNet.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import build_datasets
from src.model_builder import build_baseline_cnn, standard_callbacks

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__, " GPU:", bool(tf.config.list_physical_devices("GPU")))

## 1. Load datasets and build the model

In [ ]:
DATA_DIR = PROJECT_ROOT / "chest_xray"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
MODELS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

train_ds, val_ds, test_ds, class_weights, manifest = build_datasets(DATA_DIR)
print("Class weights:", class_weights)

In [ ]:
model = build_baseline_cnn()
model.summary()

## 2. Train

Training uses early stopping on `val_auc`. AUC is more robust than accuracy on imbalanced data.
On a CPU this notebook will take a while; on a GPU each epoch is roughly a minute.

In [ ]:
ckpt_path = str(MODELS_DIR / "baseline_cnn.keras")
callbacks = standard_callbacks(ckpt_path, patience=5)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

## 3. Training curves

In [ ]:
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(REPORTS_DIR / "baseline_history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["loss"], label="train")
axes[0].plot(hist_df["val_loss"], label="val")
axes[0].set_title("Baseline CNN: loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()

axes[1].plot(hist_df["auc"], label="train AUC")
axes[1].plot(hist_df["val_auc"], label="val AUC")
axes[1].plot(hist_df["recall"], label="train recall", linestyle="--")
axes[1].plot(hist_df["val_recall"], label="val recall", linestyle="--")
axes[1].set_title("Baseline CNN: AUC and recall")
axes[1].set_xlabel("epoch"); axes[1].legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "baseline_training_curves.png", bbox_inches="tight")
plt.show()

## 4. Test set evaluation

Quick numbers here. The full evaluation (confusion matrix, ROC, PR, threshold tuning, Grad-CAM) lives in `05_evaluation_gradcam.ipynb`, where the baseline and the transfer-learning model are compared head to head.

In [ ]:
test_metrics = model.evaluate(test_ds, return_dict=True, verbose=1)
test_metrics = {k: float(v) for k, v in test_metrics.items()}
test_metrics

In [ ]:
metrics_path = REPORTS_DIR / "baseline_test_metrics.json"
metrics_path.write_text(json.dumps(test_metrics, indent=2))
print("Saved metrics to", metrics_path)
print("Saved checkpoint to", ckpt_path)